# IMDb Genre Classifier — DistilBERT + LoRA Fine-Tuning

Fine-tune `distilbert-base-uncased` with LoRA for 5-class movie genre classification,
using the curated dataset published by the **aaiclick IMDb Dataset Builder** pipeline.

**Dataset**: [`aaiclick/imdb-wikipedia-enriched`](https://huggingface.co/datasets/aaiclick/imdb-wikipedia-enriched)  
**Model**: `distilbert-base-uncased` + LoRA (r=8)  
**Task**: Multi-class genre classification (Drama / Comedy / Action / Thriller / Horror)  
**Runtime**: ~3 min on Colab T4 GPU

---
**Why is the curated dataset better than raw IMDb?**

The aaiclick pipeline removed:
- Non-movie titles (TV series, shorts, video games, …)
- Adult content
- Movies with unknown genres or year
- Very short (< 40 min) and very long (> 300 min) edge cases
- Pre-1980 movies (sparser metadata)

Without this curation, train/test splits may contain noisy labels, near-duplicate
titles across splits, and severe class imbalance — all of which hurt model quality.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers peft torch evaluate accelerate datasets huggingface_hub scikit-learn

In [ ]:
# Cell 2 — Load curated dataset from Hugging Face
from datasets import load_dataset

ds = load_dataset("aaiclick/imdb-wikipedia-enriched", split="train")
print(f"Dataset loaded: {len(ds):,} rows")
print(f"Columns: {ds.column_names}")
print()
print(ds[0])

In [ ]:
# Cell 3 — Inspect: class distribution, text length stats

df = ds.to_pandas()

# Expand comma-separated genres to count each
all_genres = df["genres"].str.split(",").explode().str.strip()
genre_counts = all_genres.value_counts()

print("Genre distribution (all genres):")
print(genre_counts.to_string())
print()

df["title_len"] = df["primaryTitle"].str.len()
print("Title length stats:")
print(df["title_len"].describe().to_string())

In [ ]:
# Cell 4 — Simplify to top-5 genres for cleaner classification
# Each title is assigned its FIRST matching genre from the priority list.
# This avoids multi-label complexity while keeping the 5 most common genres.
from datasets import Dataset

TOP5_GENRES = ["Drama", "Comedy", "Action", "Thriller", "Horror"]
LABEL2ID = {g: i for i, g in enumerate(TOP5_GENRES)}
ID2LABEL = {i: g for g, i in LABEL2ID.items()}


def assign_label(genres_str):
    """Return the index of the first matching top-5 genre, or None."""
    genres = [g.strip() for g in genres_str.split(",")]
    for genre in TOP5_GENRES:
        if genre in genres:
            return LABEL2ID[genre]
    return None


df["label"] = df["genres"].apply(assign_label)
df_filtered = df[df["label"].notna()].copy()
df_filtered["label"] = df_filtered["label"].astype(int)

print(f"Rows after top-5 filter: {len(df_filtered):,}")
print()
print("Distribution:")
for genre, idx in LABEL2ID.items():
    count = (df_filtered["label"] == idx).sum()
    print(f"  {genre}: {count:,}")

ds5 = Dataset.from_pandas(df_filtered[["primaryTitle", "startYear", "label"]], preserve_index=False)

In [ ]:
# Cell 5 — Train / val / test split (80 / 10 / 10)
splits = ds5.train_test_split(test_size=0.2, seed=42)
val_test = splits["test"].train_test_split(test_size=0.5, seed=42)

train_ds = splits["train"]  # 80%
val_ds = val_test["train"]  # 10%
test_ds = val_test["test"]  # 10%

print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}")

In [ ]:
# Cell 6 — Tokenize using DistilBERT tokenizer
# Input text: "<primaryTitle> [year <startYear>]" so the model sees both signals.
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize(batch):
    texts = [f"{title} [year {year}]" for title, year in zip(batch["primaryTitle"], batch["startYear"], strict=False)]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=64)


train_tok = train_ds.map(tokenize, batched=True, remove_columns=["primaryTitle", "startYear"])
val_tok = val_ds.map(tokenize, batched=True, remove_columns=["primaryTitle", "startYear"])
test_tok = test_ds.map(tokenize, batched=True, remove_columns=["primaryTitle", "startYear"])

train_tok.set_format("torch")
val_tok.set_format("torch")
test_tok.set_format("torch")

print("Tokenization complete.")
print(f"Sample keys: {list(train_tok[0].keys())}")

In [ ]:
# Cell 7 — Apply LoRA adapter to DistilBERT
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForSequenceClassification

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(TOP5_GENRES),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 8 — Fine-tune with HuggingFace Trainer
# GPU (T4): ~3 min   CPU: ~20 min
import evaluate
import numpy as np
from transformers import Trainer, TrainingArguments

accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)


training_args = TrainingArguments(
    output_dir="./imdb-genre-lora",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    learning_rate=3e-4,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Cell 9 — Evaluate on held-out test set + confusion matrix
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

preds_output = trainer.predict(test_tok)
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print("Test set accuracy:", (y_pred == y_true).mean())
print()
print(classification_report(y_true, y_pred, target_names=TOP5_GENRES))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    display_labels=TOP5_GENRES,
    cmap="Blues",
    ax=ax,
)
ax.set_title("Genre Classification — Test Set Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10 — Interactive inference
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
model.to(device)


def predict_genre(title: str, year: str = "") -> str:
    """Predict the primary genre for a movie title."""
    text = f"{title} [year {year}]" if year else title
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=64).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_id = logits.argmax(dim=-1).item()
    return ID2LABEL[pred_id]


examples = [
    ("The Dark Knight", "2008"),
    ("Superbad", "2007"),
    ("Schindler's List", "1993"),
    ("Saw", "2004"),
    ("Heat", "1995"),
]

print("Inference examples:")
print(f"{'Title':<30} {'Year':<6} {'Predicted Genre'}")
print("-" * 50)
for title, year in examples:
    genre = predict_genre(title, year)
    print(f"{title:<30} {year:<6} {genre}")